# Practical 02: Data Hygiene: Missing Value Imputation, Deduplication & Renaming

**UNIT 1: Introduction to Big Data and Large Dataset Handling**

**Problem Statement:** Write a Python program to perform data hygiene on a dataset with missing values and duplicate rows. Identify missing entries using isna().sum(), demonstrate both dropping rows (dropna()) and imputing missing numerical values with fillna(mean/median), remove duplicate records using drop_duplicates(), and rename columns using rename().

In [1]:
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None, "display.width", 200)

gen = np.random.default_rng(7)
n = 160
orders = pd.DataFrame({
    "Order ID": np.arange(1001, 1001 + n),
    "Customer Name": [f"Client_{i:03d}" for i in gen.integers(1, 45, n)],
    "City": gen.choice(["Delhi", "Jaipur", "Surat", "Indore", "Nagpur", "Pune"], n),
    "Product Category": gen.choice(["Electronics", "Clothing", "Groceries", "Toys", "Books"], n),
    "Quantity": gen.integers(1, 12, n).astype(float),
    "Unit Price": np.round(gen.uniform(150, 4500, n), 2),
    "Discount %": gen.choice([0, 5, 10, 15, 25], n).astype(float),
    "Order Date": pd.date_range("2025-03-01", periods=n, freq="D").strftime("%Y-%m-%d"),
})
for col, k in [("Quantity", 10), ("Unit Price", 8), ("Discount %", 7), ("City", 6)]:
    orders.loc[gen.choice(n, k, replace=False), col] = np.nan
orders = pd.concat([orders, orders.sample(10, random_state=1)])
orders = orders.sample(frac=1, random_state=5).reset_index(drop=True)

print("Original shape:", orders.shape)
print(orders.head())

print(orders.isna().sum())

df_dropped = orders.dropna()
print(f"dropna(): {orders.shape[0]} -> {df_dropped.shape[0]} rows ({orders.shape[0] - df_dropped.shape[0]} rows lost)")

df_imputed = orders.copy()
df_imputed["Quantity"] = df_imputed["Quantity"].fillna(df_imputed["Quantity"].mean())
df_imputed["Unit Price"] = df_imputed["Unit Price"].fillna(df_imputed["Unit Price"].mean())
df_imputed["Discount %"] = df_imputed["Discount %"].fillna(df_imputed["Discount %"].mean())
df_imputed["City"] = df_imputed["City"].fillna(df_imputed["City"].mode()[0])

print("Missing values after imputation:")
print(df_imputed.isna().sum())

print("Duplicate rows found:", df_imputed.duplicated().sum())
df_clean = df_imputed.drop_duplicates().reset_index(drop=True)
print("Shape after drop_duplicates():", df_clean.shape)

df_clean = df_clean.rename(columns={
    "Order ID": "order_id",
    "Customer Name": "customer_name",
    "City": "city",
    "Product Category": "product_category",
    "Quantity": "quantity",
    "Unit Price": "unit_price",
    "Discount %": "discount_pct",
    "Order Date": "order_date",
})
print(df_clean.columns.tolist())
print(df_clean.head())

Original shape: (170, 8)

,Order ID,Customer Name,City,Product Category,Quantity,Unit Price,Discount %,Order Date
0,1115,Client_018,Delhi,Groceries,9.0,4204.70,0.0,2025-06-23
1,1055,Client_036,Jaipur,Groceries,9.0,1153.40,15.0,2025-04-24
2,1092,Client_033,Jaipur,Electronics,3.0,2950.01,15.0,2025-05-31
3,1121,Client_020,Nagpur,Toys,4.0,2730.54,15.0,2025-06-29
4,1098,Client_023,Jaipur,Books,9.0,1607.58,5.0,2025-06-06


Order ID             0
Customer Name        0
City                 6
Product Category     0
Quantity            10
Unit Price           8
Discount %           7
Order Date           0
dtype: int64
dropna(): 170 -> 142 rows (28 rows lost)
Missing values after imputation:
Order ID            0
Customer Name       0
City                0
Product Category    0
Quantity            0
Unit Price          0
Discount %          0
Order Date          0
dtype: int64
Duplicate rows found: 10
Shape after drop_duplicates(): (160, 8)
['order_id', 'customer_name', 'city', 'product_category', 'quantity', 'unit_price', 'discount_pct', 'order_date']

,order_id,customer_name,city,product_category,quantity,unit_price,discount_pct,order_date
0,1115,Client_018,Delhi,Groceries,9.0,4204.70,0.0,2025-06-23
1,1055,Client_036,Jaipur,Groceries,9.0,1153.40,15.0,2025-04-24
2,1092,Client_033,Jaipur,Electronics,3.0,2950.01,15.0,2025-05-31
3,1121,Client_020,Nagpur,Toys,4.0,2730.54,15.0,2025-06-29
4,1098,Client_023,Jaipur,Books,9.0,1607.58,5.0,2025-06-06
